# Exploration des données — DVF Paris × Transports IDF × Espaces Verts

Ce notebook réalise l'ensemble de l'analyse exploratoire du projet :

1. Chargement et nettoyage du dataset DVF
2. EDA sur les prix au m² à Paris
3. Chargement et nettoyage du dataset stations IDF
4. Croisement DVF × Transports (BallTree haversine)
5. Analyse de l'impact des transports sur le prix au m²
6. Chargement et nettoyage du dataset Espaces Verts
7. Croisement DVF × Espaces Verts (BallTree haversine)
8. Analyse de l'impact des espaces verts sur le prix au m²
9. Sauvegarde du dataset enrichi

**Sources de données :**
- `data/raw/dvf.csv` — Demandes de valeurs foncières (open data)
- `data/external/emplacement-des-gares-idf.csv` — Stations IDF (IDFM open data)
- `data/external/espaces_verts.csv` — Espaces verts de Paris (OpenData Paris)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.neighbors import BallTree

os.makedirs('../plots', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('Librairies chargées.')

## 1. Chargement et nettoyage du dataset DVF

In [ ]:
dvf_raw = pd.read_csv('../data/raw/dvf.csv', low_memory=False)
print(f'DVF brut : {dvf_raw.shape[0]:,} lignes, {dvf_raw.shape[1]} colonnes')
dvf_raw.head(2)

In [ ]:
df = dvf_raw.copy()

# Conversion des colonnes numériques (stockées en object dans le CSV brut)
df['valeur_fonciere'] = pd.to_numeric(df['valeur_fonciere'], errors='coerce')
df['surface_reelle_bati'] = pd.to_numeric(df['surface_reelle_bati'], errors='coerce')
df['code_departement'] = df['code_departement'].astype(str).str.strip()
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

n0 = len(df)

# Filtres : ventes d'appartements parisiens avec surface et prix valides
df = df[df['nature_mutation'] == 'Vente']
df = df[df['type_local'] == 'Appartement']
df = df[df['code_departement'] == '75']
df = df[df['valeur_fonciere'] > 0]
df = df[df['surface_reelle_bati'] > 0]
df = df[df['latitude'].notna() & df['longitude'].notna()]

n1 = len(df)

# Calcul du prix au m²
df['prix_m2'] = df['valeur_fonciere'] / df['surface_reelle_bati']

# Suppression des outliers extrêmes
df = df[(df['prix_m2'] >= 3000) & (df['prix_m2'] <= 25000)]
df = df.reset_index(drop=True)

n2 = len(df)

print(f'Lignes brutes              : {n0:>10,}')
print(f'Après filtrage             : {n1:>10,}')
print(f'Après suppression outliers : {n2:>10,}')
print(f'Prix m² médian             : {df["prix_m2"].median():>10,.0f} €/m²')
print(f'Prix m² moyen              : {df["prix_m2"].mean():>10,.0f} €/m²')
df[['valeur_fonciere', 'surface_reelle_bati', 'prix_m2', 'latitude', 'longitude']].head(3)

## 2. EDA — Prix au m² à Paris

In [ ]:
# Distribution des prix au m²
fig, ax = plt.subplots(figsize=(11, 5))

ax.hist(df['prix_m2'], bins=80, color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(df['prix_m2'].median(), color='tomato', linestyle='--', linewidth=1.5,
           label=f'Médiane : {df["prix_m2"].median():,.0f} €/m²')
ax.axvline(df['prix_m2'].mean(), color='orange', linestyle='--', linewidth=1.5,
           label=f'Moyenne : {df["prix_m2"].mean():,.0f} €/m²')

ax.set_xlabel('Prix au m² (€/m²)')
ax.set_ylabel('Nombre de transactions')
ax.set_title('Distribution des prix au m² — Appartements Paris')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,} €'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()

plt.tight_layout()
plt.savefig('../plots/01_distribution_prix_m2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé : plots/01_distribution_prix_m2.png')

In [ ]:
# Prix médian par arrondissement
df['code_postal'] = df['code_postal'].astype(str).str.strip()
median_by_arr = (
    df.groupby('code_postal')['prix_m2']
    .median()
    .sort_values(ascending=False)
)

print('=== Top 5 arrondissements les plus chers ===')
print(median_by_arr.head(5).apply(lambda x: f'{x:,.0f} €/m²').to_string())
print()
print('=== Top 5 arrondissements les moins chers ===')
print(median_by_arr.tail(5).apply(lambda x: f'{x:,.0f} €/m²').to_string())

In [ ]:
# Bar chart : prix m² médian par arrondissement
arr_sorted = median_by_arr.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(arr_sorted.index, arr_sorted.values, color='steelblue', edgecolor='white')

ax.set_xlabel('Prix m² médian (€/m²)')
ax.set_title('Prix m² médian par arrondissement — Appartements Paris')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,} €'))

for bar, val in zip(bars, arr_sorted.values):
    ax.text(val + 50, bar.get_y() + bar.get_height() / 2,
            f'{int(val):,} €', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('../plots/01_prix_m2_par_arrondissement.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé : plots/01_prix_m2_par_arrondissement.png')

In [ ]:
# Distribution des surfaces
surface_filtre = df[df['surface_reelle_bati'] <= 200]

fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(surface_filtre['surface_reelle_bati'], bins=60,
        color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(df['surface_reelle_bati'].median(), color='tomato', linestyle='--', linewidth=1.5,
           label=f'Médiane : {df["surface_reelle_bati"].median():.0f} m²')

ax.set_xlabel('Surface réelle bâtie (m²)')
ax.set_ylabel('Nombre de transactions')
ax.set_title('Distribution des surfaces — Appartements Paris (≤ 200 m²)')
ax.legend()

plt.tight_layout()
plt.savefig('../plots/01_distribution_surface.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé : plots/01_distribution_surface.png')

In [ ]:
# Prix m² médian par nombre de pièces
df['nombre_pieces_principales'] = pd.to_numeric(df['nombre_pieces_principales'], errors='coerce')
pieces_valides = df[df['nombre_pieces_principales'].between(1, 7)]

median_by_pieces = (
    pieces_valides.groupby('nombre_pieces_principales')['prix_m2']
    .median()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(median_by_pieces.index.astype(int), median_by_pieces.values,
              color='steelblue', edgecolor='white')
ax.set_xlabel('Nombre de pièces principales')
ax.set_ylabel('Prix m² médian (€/m²)')
ax.set_title('Prix m² médian par nombre de pièces — Appartements Paris')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,} €'))

for bar, val in zip(bars, median_by_pieces.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f'{int(val):,} €', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../plots/01_prix_m2_par_nb_pieces.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé : plots/01_prix_m2_par_nb_pieces.png')

In [ ]:
# Statistiques descriptives globales
print('=== Statistiques descriptives — prix au m² ===')
print(df['prix_m2'].describe().round(0).astype(int).to_string())
print()
print('=== Statistiques descriptives — surface réelle bâtie ===')
print(df['surface_reelle_bati'].describe().round(1).to_string())

## 3. Chargement et nettoyage du dataset Stations IDF

Source : Île-de-France Mobilités (IDFM) — open data  
Fichier : `data/external/emplacement-des-gares-idf.csv`

In [ ]:
stations_raw = pd.read_csv(
    '../data/external/emplacement-des-gares-idf.csv',
    sep=';',
    low_memory=False
)
print(f'Stations brut : {stations_raw.shape[0]:,} lignes, {stations_raw.shape[1]} colonnes')
print(f'Colonnes : {list(stations_raw.columns)}')
stations_raw.head(2)

In [ ]:
stations = stations_raw.copy()

# La colonne "Geo Point" contient "latitude, longitude" en texte
coords = stations['Geo Point'].str.split(',', expand=True)
stations['latitude_station'] = pd.to_numeric(coords[0], errors='coerce')
stations['longitude_station'] = pd.to_numeric(coords[1], errors='coerce')

stations = stations[['nom_long', 'res_com', 'indice_lig', 'mode', 'exploitant',
                      'latitude_station', 'longitude_station']]
stations = stations.dropna(subset=['latitude_station', 'longitude_station'])
stations = stations.reset_index(drop=True)

print(f'Stations nettoyées : {len(stations):,}')
print(f'Modes de transport : {sorted(stations["mode"].dropna().unique())}')
stations.head(3)

## 4. Croisement DVF × Transports — BallTree (Haversine)

On utilise `sklearn.neighbors.BallTree` avec la métrique haversine pour calculer les distances sphériques de façon vectorisée, sans boucle Python. Les coordonnées sont converties en radians avant d'alimenter le BallTree.

In [ ]:
EARTH_RADIUS_KM = 6371.0

dvf_coords_rad = np.radians(df[['latitude', 'longitude']].values)
stations_coords_rad = np.radians(stations[['latitude_station', 'longitude_station']].values)

tree_transport = BallTree(stations_coords_rad, metric='haversine')
print('BallTree transports construit.')

In [ ]:
# Station la plus proche : distance, nom, mode, ligne
distances_rad, indices = tree_transport.query(dvf_coords_rad, k=1)

distances_m = distances_rad[:, 0] * EARTH_RADIUS_KM * 1000
nearest_idx = indices[:, 0]

df['distance_station_plus_proche'] = distances_m
df['nom_station_plus_proche'] = stations.loc[nearest_idx, 'nom_long'].values
df['mode_station_plus_proche'] = stations.loc[nearest_idx, 'mode'].values
df['ligne_station_plus_proche'] = stations.loc[nearest_idx, 'indice_lig'].values

print('Station la plus proche calculée.')
df[['distance_station_plus_proche', 'nom_station_plus_proche',
    'mode_station_plus_proche', 'ligne_station_plus_proche']].head(5)

In [ ]:
# Nombre de stations dans un rayon de 500m et 1000m
radius_500m_rad = 500 / (EARTH_RADIUS_KM * 1000)
radius_1000m_rad = 1000 / (EARTH_RADIUS_KM * 1000)

df['nb_stations_500m'] = tree_transport.query_radius(dvf_coords_rad, r=radius_500m_rad, count_only=True)
df['nb_stations_1000m'] = tree_transport.query_radius(dvf_coords_rad, r=radius_1000m_rad, count_only=True)

print(f'Médiane nb_stations_500m  : {df["nb_stations_500m"].median():.0f}')
print(f'Médiane nb_stations_1000m : {df["nb_stations_1000m"].median():.0f}')
df[['nb_stations_500m', 'nb_stations_1000m']].describe()

## 5. EDA — Impact des transports sur le prix au m²

In [ ]:
# Résumé : distance à la station la plus proche
print('=== Distance à la station la plus proche (m) ===')
print(df['distance_station_plus_proche'].describe().round(1))

mask_500 = df['distance_station_plus_proche'] < 500
pct_500 = mask_500.mean() * 100
prix_moins_500 = df.loc[mask_500, 'prix_m2'].median()
prix_plus_500 = df.loc[~mask_500, 'prix_m2'].median()

print(f'\nBiens à moins de 500m : {mask_500.sum():,} ({pct_500:.1f}%)')
print(f'Prix m² médian < 500m : {prix_moins_500:,.0f} €/m²')
print(f'Prix m² médian > 500m : {prix_plus_500:,.0f} €/m²')
print(f'Écart médian          : {prix_moins_500 - prix_plus_500:+,.0f} €/m²')

In [ ]:
# Prix m² médian par tranche de distance aux transports
bins = [0, 250, 500, 750, 1000, np.inf]
labels_tranches = ['0–250m', '250–500m', '500–750m', '750–1000m', '+1000m']

df['tranche_distance'] = pd.cut(
    df['distance_station_plus_proche'], bins=bins, labels=labels_tranches
)

median_by_tranche = df.groupby('tranche_distance', observed=True)['prix_m2'].median()
count_by_tranche = df.groupby('tranche_distance', observed=True)['prix_m2'].count()

print('=== Prix m² médian par tranche de distance ===')
print(pd.DataFrame({
    'prix_m2_median': median_by_tranche.round(0).astype(int),
    'nb_biens': count_by_tranche
}).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels_tranches, median_by_tranche.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Tranche de distance à la station')
ax.set_ylabel('Prix m² médian (€/m²)')
ax.set_title('Prix m² médian par tranche de distance à la station de transport')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,} €'))

for bar, val in zip(bars, median_by_tranche.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f'{int(val):,} €', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../plots/02_prix_m2_par_tranche_distance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé : plots/02_prix_m2_par_tranche_distance.png')

In [ ]:
# Prix m² médian selon le mode de transport le plus proche
median_by_mode = (
    df.groupby('mode_station_plus_proche')['prix_m2']
    .median()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(median_by_mode.index, median_by_mode.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Mode de transport le plus proche')
ax.set_ylabel('Prix m² médian (€/m²)')
ax.set_title('Prix m² médian selon le mode de transport le plus proche')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,} €'))
plt.xticks(rotation=20, ha='right')

for bar, val in zip(bars, median_by_mode.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f'{int(val):,} €', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../plots/02_prix_m2_par_mode_transport.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé : plots/02_prix_m2_par_mode_transport.png')

## 6. Chargement et nettoyage du dataset Espaces Verts Paris

Source : OpenData Paris — Espaces verts de la Ville de Paris  
Fichier : `data/external/espaces_verts.csv`

On utilise la colonne `Geo point` (latitude, longitude) et `Surface calculée` (surface en m²).

In [ ]:
ev_raw = pd.read_csv(
    '../data/external/espaces_verts.csv',
    sep=';',
    low_memory=False
)
print(f'Espaces verts brut : {ev_raw.shape[0]:,} lignes, {ev_raw.shape[1]} colonnes')
print(f'Colonnes disponibles : {list(ev_raw.columns[:10])} ...')
ev_raw[['Nom de l\'espace vert', 'Typologie d\'espace vert', 'Surface calculée', 'Geo point']].head(3)

In [ ]:
ev = ev_raw.copy()

# Extraction latitude / longitude depuis la colonne "Geo point"
coords_ev = ev['Geo point'].str.split(',', expand=True)
ev['latitude_ev'] = pd.to_numeric(coords_ev[0], errors='coerce')
ev['longitude_ev'] = pd.to_numeric(coords_ev[1], errors='coerce')

# Conversion de la surface
ev['surface_ev'] = pd.to_numeric(ev['Surface calculée'], errors='coerce')

# Garder uniquement les colonnes utiles
ev = ev[['Nom de l\'espace vert', 'Typologie d\'espace vert',
         'surface_ev', 'latitude_ev', 'longitude_ev']]
ev.columns = ['nom_ev', 'type_ev', 'surface_ev', 'latitude_ev', 'longitude_ev']

# Supprimer les lignes sans coordonnées
ev = ev.dropna(subset=['latitude_ev', 'longitude_ev'])
ev = ev.reset_index(drop=True)

print(f'Espaces verts nettoyés : {len(ev):,}')
print(f'Types : {ev["type_ev"].value_counts().head(5).to_dict()}')
ev.head(3)

## 7. Croisement DVF × Espaces Verts — BallTree (Haversine)

In [ ]:
ev_coords_rad = np.radians(ev[['latitude_ev', 'longitude_ev']].values)

tree_ev = BallTree(ev_coords_rad, metric='haversine')
print('BallTree espaces verts construit.')

In [ ]:
# Espace vert le plus proche : distance et surface
dist_ev_rad, idx_ev = tree_ev.query(dvf_coords_rad, k=1)

dist_ev_m = dist_ev_rad[:, 0] * EARTH_RADIUS_KM * 1000
nearest_ev_idx = idx_ev[:, 0]

df['distance_ev_plus_proche'] = dist_ev_m
df['nom_ev_plus_proche'] = ev.loc[nearest_ev_idx, 'nom_ev'].values
df['surface_ev_plus_proche'] = ev.loc[nearest_ev_idx, 'surface_ev'].values

# Nombre d'espaces verts dans un rayon de 500m
df['nb_ev_500m'] = tree_ev.query_radius(dvf_coords_rad, r=radius_500m_rad, count_only=True)

print('Features espaces verts calculées.')
print(f'Distance médiane à l\'espace vert le plus proche : {df["distance_ev_plus_proche"].median():.0f} m')
print(f'Médiane nb_ev_500m : {df["nb_ev_500m"].median():.0f}')
df[['distance_ev_plus_proche', 'nom_ev_plus_proche', 'surface_ev_plus_proche', 'nb_ev_500m']].head(5)

## 8. EDA — Impact des espaces verts sur le prix au m²

In [ ]:
# Corrélation distance espace vert / prix m²
corr_ev = df['distance_ev_plus_proche'].corr(df['prix_m2'])
print(f'Corrélation (Pearson) distance_ev / prix_m2 : {corr_ev:.4f}')

# Prix médian selon la proximité à un espace vert (seuil 300m)
mask_ev_300 = df['distance_ev_plus_proche'] < 300
prix_pres_ev = df.loc[mask_ev_300, 'prix_m2'].median()
prix_loin_ev = df.loc[~mask_ev_300, 'prix_m2'].median()

print(f'\nBiens à moins de 300m d\'un espace vert : {mask_ev_300.sum():,} ({mask_ev_300.mean()*100:.1f}%)')
print(f'Prix m² médian < 300m : {prix_pres_ev:,.0f} €/m²')
print(f'Prix m² médian > 300m : {prix_loin_ev:,.0f} €/m²')
print(f'Écart médian          : {prix_pres_ev - prix_loin_ev:+,.0f} €/m²')

In [ ]:
# Prix m² médian par tranche de distance à l'espace vert le plus proche
bins_ev = [0, 100, 200, 300, 500, np.inf]
labels_ev = ['0–100m', '100–200m', '200–300m', '300–500m', '+500m']

df['tranche_distance_ev'] = pd.cut(
    df['distance_ev_plus_proche'], bins=bins_ev, labels=labels_ev
)

median_ev_tranche = df.groupby('tranche_distance_ev', observed=True)['prix_m2'].median()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels_ev, median_ev_tranche.values, color='mediumseagreen', edgecolor='white')
ax.set_xlabel('Tranche de distance à l\'espace vert le plus proche')
ax.set_ylabel('Prix m² médian (€/m²)')
ax.set_title('Prix m² médian par tranche de distance à l\'espace vert le plus proche')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,} €'))

for bar, val in zip(bars, median_ev_tranche.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f'{int(val):,} €', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../plots/03_prix_m2_par_distance_espace_vert.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé : plots/03_prix_m2_par_distance_espace_vert.png')

## 9. Sauvegarde du dataset enrichi

In [ ]:
output_path = '../data/processed/dvf_paris_enriched.csv'
df.to_csv(output_path, index=False)

print(f'Dataset enrichi sauvegardé : {output_path}')
print(f'Shape finale : {df.shape}')
print()
print('Features ajoutées :')
new_cols = [
    'distance_station_plus_proche', 'nom_station_plus_proche',
    'mode_station_plus_proche', 'ligne_station_plus_proche',
    'nb_stations_500m', 'nb_stations_1000m',
    'distance_ev_plus_proche', 'nom_ev_plus_proche',
    'surface_ev_plus_proche', 'nb_ev_500m'
]
print(df[new_cols].describe())